In [0]:
%load_ext autoreload
%autoreload 2

# Treinamento do modelo:

Este notebook tem como objetivo treinar um modelo de Rede Neural Convolucional (CNN) e conduzir experimentos utilizando o MLflow, com a finalidade de identificar e selecionar o modelo com melhor desempenho.

Serão testados:

- **SimpleCNN:**

    Modelo criado do zero para o problema em questão.

- **ResNet-18:**

    Rede neural convolucional profunda pré treinada com 18 camadas.

- **EfficientNet-b0:**

    Faz parte da família EfficientNet desenvolvida pelo Google, sendo b0 um modelo leve e de alta eficiência.

- **DenseNet-121:**

    Rede neural convolucional profunda, famosa pela arquitetura densamente conectada. Opera melhor com bases de dados maiores, será testada apenas como experimento.

Para efeitos de comparação, em cada run serão logados os seguintes parametros:

- Nome do modelo;
- Taxa de aprendizado;
- Freeze Backbone;
- Épocas;
- Classes;
- Early stopping (caso ocorra)

Para avaliação do desempenho serão logadas as métricas train_loss, train_acc, val_loss, val_acc.

Para cada modelo, serão treinadas duas versões distintas. A primeira utilizará o learning rate definido a partir do lr_finder, enquanto a segunda será treinada aplicando a estratégia One Cycle Policy. O objetivo é comparar o desempenho entre as duas abordagens e selecionar a configuração que apresentar os melhores resultados.

In [0]:
import mlflow
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

import torch
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from src.train import find_best_lr, run_training
from src.dataset.milho_dataset import get_dataloaders_milho
from src.evaluate import get_predictions
from src.models.milho_models import get_model

## Modelo criado do zero:

#### Learning Rate Finder CNN:

In [0]:
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
train_loader, _, _ , classes = get_dataloaders_milho(data_dir=data_dir)
model_name = "SimpleCNN"
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
freeze_backbone = True

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="SimpleCNN_best_lr"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("num_classes", num_classes)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("classes", classes)

    suggested_lr, fig = find_best_lr(
                model_name=model_name,
                num_classes=num_classes,
                train_loader=train_loader,
                device=device,
                freeze_backbone=freeze_backbone
                )

    mlflow.log_param("suggested_lr", suggested_lr)

    fig

In [0]:
run_id = "085eca5d71904b6dbf0c9339df7b3d9f"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr = float(params.get("suggested_lr"))
suggested_lr

#### CNN Learning Rate fixo:

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "SimpleCNN"
learning_rate = suggested_lr
epochs = 30
freeze_backbone = True
patience = 5
use_onecycle = False
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="SimpleCNN"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("patience", patience)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_CNN, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        patience=patience
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_CNN(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_CNN,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

## Matriz confusão CNN LR fixo

In [0]:
run_id_CNN = "6d60be5f39024202b9fb0863811c8632"

with mlflow.start_run(run_id=run_id_CNN):

    val_labels, val_preds = get_predictions(model_CNN, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_Simple_CNN.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_Simple_CNN.png")

## CNN 1cycle policy

In [0]:
run_id = "085eca5d71904b6dbf0c9339df7b3d9f"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr = float(params.get("suggested_lr"))

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "SimpleCNN"
learning_rate = suggested_lr
epochs = 25
freeze_backbone = True
use_onecycle = True
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="SimpleCNN_1cycle"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_CNN_1cycle, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        use_onecycle=use_onecycle
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_CNN_1cycle(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_CNN_1cycle,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

## Matriz confusão CNN 1cycle policy

In [0]:
run_id_CNN_1cycle = "39efe20dccaf4baf9c33e4e128c93067"

with mlflow.start_run(run_id=run_id_CNN_1cycle):

    val_labels, val_preds = get_predictions(model_CNN_1cycle, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_Simple_CNN_1cycle.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_Simple_CNN_1cycle.png")

## LR finder Resnet18

In [0]:
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
train_loader, _, _ , classes = get_dataloaders_milho(data_dir=data_dir)
model_name = "resnet18"
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
freeze_backbone = True

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="resnet18_best_lr"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("num_classes", num_classes)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("classes", classes)

    suggested_lr, fig = find_best_lr(
                model_name=model_name,
                num_classes=num_classes,
                train_loader=train_loader,
                device=device,
                freeze_backbone=freeze_backbone
                )

    mlflow.log_param("suggested_lr", suggested_lr)

    fig

## Resnet18 LR fixo

In [0]:
run_id = "97cb6892487643f28c4c42bcf3b198be"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr_resnet = float(params.get("suggested_lr")) / 10
suggested_lr_resnet

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "resnet18"
learning_rate = suggested_lr_resnet
epochs = 30
freeze_backbone = True
patience = 5
use_onecycle = False
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="resnet18"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("patience", patience)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_resnet18, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        patience=patience
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_resnet18(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_resnet18,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

## Matriz confusão Resnet18 LR fixo

In [0]:
run_id_resnet18 = "ecfe31e2a87c42cab69b824c7e7b4fa2"

with mlflow.start_run(run_id=run_id_resnet18):

    val_labels, val_preds = get_predictions(model_resnet18, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_resnet18.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_resnet18.png")


## Resnet18 1cycle policy

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "resnet18"
learning_rate = suggested_lr_resnet
epochs = 30
freeze_backbone = True
use_onecycle = True
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="resnet18_1cycle"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_resnet18_1cycle, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        use_onecycle=use_onecycle
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_resnet18_1cycle(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_resnet18_1cycle,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

In [0]:
run_id_resnet18_1cycle = "67254b3ec1ec4a0b97a5835eafaee785"
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
model_resnet18_1cycle = mlflow.pytorch.load_model(f"runs:/{run_id_resnet18_1cycle}/model")
_, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with mlflow.start_run(run_id=run_id_resnet18_1cycle):

    val_labels, val_preds = get_predictions(model_resnet18_1cycle, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_resnet18_1cycle.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_resnet18_1cycle.png")

## LR finder efficientnet b0

In [0]:
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
train_loader, _, _ , classes = get_dataloaders_milho(data_dir=data_dir)
model_name = "efficientnet_b0"
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
freeze_backbone = True

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="efficientnet_b0_best_lr"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("num_classes", num_classes)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("classes", classes)

    suggested_lr, fig = find_best_lr(
                model_name=model_name,
                num_classes=num_classes,
                train_loader=train_loader,
                device=device,
                freeze_backbone=freeze_backbone
                )

    mlflow.log_param("suggested_lr", suggested_lr)

    fig

## Efficientnet_b0 LR fixo

In [0]:
run_id = "9d6edde8c6694ea5a888ab70dcf04695"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr_effnet = float(params.get("suggested_lr"))
suggested_lr_effnet

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "efficientnet_b0"
learning_rate = suggested_lr_effnet
epochs = 30
freeze_backbone = True
patience = 5
use_onecycle = False
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="efficientnet_b0"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("patience", patience)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_effnetb0, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        patience=patience
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_effnetb0(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_effnetb0,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

## Matriz confusão Efficientnet_b0 LR fixo

In [0]:
run_id_effnet = "a6af598f96c243e9b60e29937eb6aec5"

with mlflow.start_run(run_id=run_id_effnet):

    val_labels, val_preds = get_predictions(model_effnetb0, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_efnet.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_efnet.png")

## Efficientnet_b0 1cycle_policy

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "efficientnet_b0"
learning_rate = suggested_lr_effnet
epochs = 30
freeze_backbone = True
use_onecycle = True
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="efficientnet_b0_1cycle"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_effnetb0_1cycle, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        use_onecycle=use_onecycle
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_effnetb0_1cycle(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_effnetb0_1cycle,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

## Matriz confusão Efficientnet 1cycle

In [0]:
run_id_effnet_1cycle = "e3f855c9716c47dab3f623a344e6ae05"

with mlflow.start_run(run_id=run_id_effnet_1cycle):

    val_labels, val_preds = get_predictions(model_effnetb0_1cycle, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_efnet_1cycle.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_efnet_1cycle.png")

## LR finder Densenet121

In [0]:
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
train_loader, _, _ , classes = get_dataloaders_milho(data_dir=data_dir)
model_name = "densenet121"
num_classes = len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
freeze_backbone = True

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="densenet121_best_lr"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("num_classes", num_classes)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("classes", classes)

    suggested_lr, fig = find_best_lr(
                model_name=model_name,
                num_classes=num_classes,
                train_loader=train_loader,
                device=device,
                freeze_backbone=freeze_backbone
                )

    mlflow.log_param("suggested_lr", suggested_lr)

    fig

## Densenet121 LR fixo

In [0]:
run_id = "9375f9c7b8284e458de6969fabda6a0e"
client = MlflowClient()
run = client.get_run(run_id)
params = run.data.params
suggested_lr_densenet = float(params.get("suggested_lr"))
suggested_lr_densenet

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "densenet121"
learning_rate = suggested_lr_densenet
epochs = 30
freeze_backbone = True
patience = 5
use_onecycle = False
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="densenet121"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("patience", patience)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_densenet, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        patience=patience
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_densenet(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_densenet,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

## Matriz confusão densenet121 lr fixo:

In [0]:
run_id_densenet = "aafaa31d71f5460d8cb78214cd97f94d"

with mlflow.start_run(run_id=run_id_densenet):

    val_labels, val_preds = get_predictions(model_densenet, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_dense.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_dense.png")

## Densenet121 1cycle policy

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "densenet121"
learning_rate = suggested_lr_densenet
epochs = 30
freeze_backbone = True
use_onecycle = True
data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

train_loader, val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

numero_classes = len(classes)

mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="densenet121_1cycle"):

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("freeze_backbone", freeze_backbone)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("classes", classes)
    mlflow.log_param("use_onecycle", use_onecycle)

    model_densenet_1cycle, history = run_training(
        model_name=model_name,
        num_classes=numero_classes,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        freeze_backbone=freeze_backbone,
        device=device,
        use_onecycle=use_onecycle
    )

    for epoch in range(len(history["train_loss"])):
        mlflow.log_metric("train_loss", history["train_loss"][epoch], step=epoch)
        mlflow.log_metric("train_acc", history["train_acc"][epoch], step=epoch)
        mlflow.log_metric("val_loss", history["val_loss"][epoch], step=epoch)
        mlflow.log_metric("val_acc", history["val_acc"][epoch], step=epoch)

    example_input, _ = next(iter(train_loader))

    with torch.no_grad():
        example_output = model_densenet_1cycle(example_input.to(device))

    signature = infer_signature(
        example_input.cpu().numpy(),
        example_output.cpu().numpy()
    )

    mlflow.pytorch.log_model(
        model_densenet_1cycle,
        "model",
        signature=signature,
        input_example=example_input[:1].cpu().numpy()
    )

## Matriz confusão densenet121 1cycle policy

In [0]:
run_id_densenet_1cycle = "5af183c6045c4698816dd0f6d711bc4e"
artifact_path = "model"
model_uri = f"runs:/{run_id_densenet_1cycle}/{artifact_path}"
model_densenet_1cycle = mlflow.pytorch.load_model(model_uri)

data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
_ , val_loader, _ , classes = get_dataloaders_milho(
    data_dir=data_dir
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

with mlflow.start_run(run_id=run_id_densenet_1cycle):


    val_labels, val_preds = get_predictions(model_densenet_1cycle, val_loader, device)

    cm = confusion_matrix(val_labels, val_preds)

    plt.figure(figsize=(8,6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=classes,
        yticklabels=classes
    )
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title("Matriz de Confusão - Validação")

    plt.tight_layout()
    plt.savefig("figures/confusion_matrix_dense_1cycle.png")
    plt.close()

    mlflow.log_artifact("figures/confusion_matrix_dense_1cycle.png")

In [0]:
experiment_name = "/Shared/Milho_Doencas"

experiment = mlflow.get_experiment_by_name(experiment_name)

runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

df_compare = runs[[
    "run_id",
    "params.model_name",
    "params.learning_rate",
    "params.use_onecycle",
    "params.epochs",
    "metrics.train_acc",
    "metrics.val_acc",
    "metrics.train_loss",
    "metrics.val_loss"
]].copy()

df_compare.columns = [
    "run_id",
    "model",
    "lr",
    "onecycle",
    "epochs",
    "train_acc",
    "val_acc",
    "train_loss",
    "val_loss"
]

df_compare["score"] = (
    df_compare["val_acc"] - 0.1 * df_compare["val_loss"]
)

df_compare = df_compare.sort_values(
    by="score",
    ascending=False
)

df_compare.head(10)

In [0]:
best_run = df_compare.iloc[0]

print("🏆 Melhor modelo encontrado:")
print(best_run)